# 1. Prepare the experimental inputs / 실험 입력 생성
Read MIMIC-IV 3.1 and eICU-CRD 2.0, construct culture episodes, split patients, and create identical four-AST inputs and nested support sets for all methods.

원자료에서 배양 사례·환자 분할·기존약 네 개 관측·중첩 support와 query를 생성합니다. 목표약과 개발약은 기존약 사전학습에서 제외합니다. 설정 셀 이후의 각 소스 셀은 해당 단계의 전체 구현이며, 실행 셀에서 순서대로 호출합니다.

In [ ]:
from pathlib import Path
import os, json
PACKAGE = Path.cwd().resolve()
assert (PACKAGE/'config/protocol.json').is_file(), 'Open this notebook from the CDLD_AST directory.'
OUTPUT = Path(os.environ.get('CDLD_OUTPUT_DIR', str(Path.home()/'CDLD_AST_outputs'))).expanduser().resolve()
assert OUTPUT != PACKAGE and PACKAGE not in OUTPUT.parents, 'Keep generated files outside the code directory.'
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN = OUTPUT/'run'
RUN.mkdir(exist_ok=True)
import shutil, time, contextlib, io
import numpy as np
import pandas as pd
from IPython.display import display
RAW_DIR = Path(os.environ.get('CDLD_RAW_DIR', str(Path.home()/'Downloads'))).expanduser().resolve()
FILES = ['microbiologyevents','icustays','patients','admissions','microLab','patient']
assert all((RAW_DIR/(f+'.csv.gz')).is_file() for f in FILES), 'Prepare the six source files listed in README.md.'
GENERATED = OUTPUT
STAGES = {1: OUTPUT/'preprocessing/cohort', 2: OUTPUT/'preprocessing/splits',
          117: OUTPUT/'preprocessing/development', 118: OUTPUT/'preprocessing/training',
          119: OUTPUT/'preprocessing/evaluation', 128: OUTPUT/'preprocessing/classical'}
for folder in STAGES.values(): folder.mkdir(parents=True, exist_ok=True)
shutil.copy2(PACKAGE/'config/development_tasks.json', STAGES[117]/'feasibility_proposal.json')
shutil.copy2(PACKAGE/'config/input_protocol.json', STAGES[118]/'frozen_protocol.json')
INPUT_SOURCES = {}

## AST cohort construction / AST 코호트 구성

In [ ]:
INPUT_SOURCES['cohort.py'] = r'''
"""AST cohort construction / AST 코호트 구성"""
from pathlib import Path
import hashlib
import json
import time
import pandas as pd

HOME = Path(__file__).resolve().parents[1]
SOURCE = Path.home() / 'Downloads'
OUTPUT = HOME / 'preprocessing/cohort'
SPECIES = {'ESCHERICHIA COLI': 'ecoli', 'KLEBSIELLA PNEUMONIAE': 'kpneumoniae',
           'PSEUDOMONAS AERUGINOSA': 'paeruginosa', 'STAPH AUREUS COAG +': 'saureus',
           'STAPHYLOCOCCUS AUREUS': 'saureus'}
ALIASES = {'meropenam':'meropenem', 'trimethoprim/sulfa':'trimethoprim/sulfamethoxazole',
           'piperacillin/tazo':'piperacillin/tazobactam',
           'amoxicillin/clavulonic acid':'amoxicillin/clavulanic acid',
           'ticarcillin/clavulonic acid':'ticarcillin/clavulanic acid'}
AUDIT = {}


def sha256(path):
    value = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()


def drug_names(series):
    return series.str.strip().str.lower().replace(ALIASES)


def episode_ids(frame, columns, prefix):
                                                                             
    return prefix + frame[columns].astype(str).agg('|'.join, axis=1)


def finalize(frame, dataset):
    counts = frame.groupby(['episode_id', 'drug']).sir.nunique()
    conflicts = counts[counts > 1].index
    rejected = frame.set_index(['episode_id', 'drug']).index.isin(conflicts)
    audit = {'input_rows':len(frame), 'conflicting_cells':len(conflicts),
             'conflicting_rows':int(rejected.sum())}
    frame = frame.loc[~rejected].drop_duplicates(['episode_id', 'drug']).copy()
    audit['unique_cells_before_minimum'] = len(frame)
    sizes = frame.groupby('episode_id').drug.transform('nunique')
    frame = frame[sizes >= 5].copy()
    frame['y'] = (frame.sir == 'S').astype(int)
    assert not frame.duplicated(['episode_id', 'drug']).any()
    assert frame.groupby('episode_id').patient_id.nunique().max() == 1
    assert frame.sir.isin(['S','I','R']).all()
    AUDIT[dataset] = audit
    return frame.sort_values(['episode_id','drug'])


def mimic():
    stays = pd.read_csv(SOURCE / 'icustays.csv.gz')
    patients = pd.read_csv(SOURCE / 'patients.csv.gz', usecols=['subject_id','anchor_age','anchor_year'])
    admissions = pd.read_csv(SOURCE / 'admissions.csv.gz', usecols=['hadm_id','admittime','dischtime'])
    stays = stays.merge(patients, on='subject_id', validate='many_to_one').merge(admissions,on='hadm_id',validate='many_to_one')
    for column in ['intime','outtime','admittime','dischtime']:
        stays[column] = pd.to_datetime(stays[column])
    selected = []
    tally = {'all_rows':0,'four_species_sir_rows':0,'missing_charttime':0,'missing_key_rows':0}
    columns = ['microevent_id','subject_id','hadm_id','micro_specimen_id','test_itemid',
               'org_itemid','isolate_num','charttime','org_name','ab_name','interpretation','spec_type_desc']
    key = ['subject_id','hadm_id','micro_specimen_id','test_itemid','org_itemid','isolate_num']
    for index, chunk in enumerate(pd.read_csv(SOURCE/'microbiologyevents.csv.gz', usecols=columns, chunksize=250000)):
        tally['all_rows'] += len(chunk)
        chunk['species'] = chunk.org_name.map(SPECIES)
        chunk = chunk[chunk.species.notna() & chunk.interpretation.isin(['S','I','R'])].copy()
        tally['four_species_sir_rows'] += len(chunk)
        tally['missing_charttime'] += int(chunk.charttime.isna().sum())
        tally['missing_key_rows'] += int(chunk[key].isna().any(axis=1).sum())
        chunk = chunk.dropna(subset=['charttime',*key])
        chunk['charttime'] = pd.to_datetime(chunk.charttime)
        chunk = chunk.merge(stays, on=['subject_id','hadm_id'], how='inner')
        age = chunk.anchor_age + chunk.charttime.dt.year - chunk.anchor_year
        allowed = ((age >= 18) & (chunk.charttime >= chunk.intime-pd.Timedelta(hours=24)) &
                   (chunk.charttime <= chunk.outtime) & (chunk.charttime >= chunk.admittime) &
                   (chunk.charttime <= chunk.dischtime))
        chunk = chunk[allowed].copy()
        chunk['pre_icu'] = chunk.charttime < chunk.intime
        chunk['distance'] = (chunk.charttime-chunk.intime).abs()
        chunk = chunk.sort_values(['pre_icu','distance','stay_id']).drop_duplicates('microevent_id')
        for column in key:
            chunk[column] = chunk[column].astype('int64')
        chunk['episode_id'] = episode_ids(chunk,key,'m:')
        chunk['patient_id'] = 'm:' + chunk.subject_id.astype(str)
        chunk['drug'] = drug_names(chunk.ab_name)
        chunk = chunk[chunk.drug.notna() & ~chunk.drug.isin(['','other'])]
        chunk['sir'] = chunk.interpretation
        chunk['site'] = chunk.spec_type_desc
        chunk['hospital_id'] = 'mimic'
        selected.append(chunk[['episode_id','patient_id','species','drug','sir','site','hospital_id','pre_icu']])
        print(f'MIMIC chunk {index+1}: source={tally["all_rows"]:,}, selected={sum(map(len,selected)):,}',flush=True)
    result = finalize(pd.concat(selected,ignore_index=True),'mimic')
    AUDIT['mimic']['selection'] = tally
    return result


def eicu():
    frame = pd.read_csv(SOURCE/'microLab.csv.gz')
    patients = pd.read_csv(SOURCE/'patient.csv.gz',dtype={'age':str,'uniquepid':str})
    frame = frame.merge(patients[['patientunitstayid','uniquepid','age','hospitalid','hospitaladmitoffset','unitdischargeoffset']],on='patientunitstayid',validate='many_to_one')
    frame['species'] = frame.organism.str.upper().map(SPECIES)
    frame['sir'] = frame.sensitivitylevel.str.strip().str.upper().replace({'SENSITIVE':'S','INTERMEDIATE':'I','RESISTANT':'R','SUSCEPTIBLE':'S'})
    frame['drug'] = drug_names(frame.antibiotic)
    age = pd.to_numeric(frame.age.replace({'> 89':'90','>89':'90'}),errors='coerce')
    valid = ((age >=18) & frame.species.notna() & frame.sir.isin(['S','I','R']) &
             frame.drug.notna() & ~frame.drug.isin(['','other']) &
             (frame.culturetakenoffset >= -1440) & (frame.culturetakenoffset >= frame.hospitaladmitoffset) &
             (frame.culturetakenoffset <= frame.unitdischargeoffset))
    frame = frame[valid].dropna(subset=['uniquepid','culturesite','culturetakenoffset']).copy()
    frame['episode_id'] = episode_ids(frame,['patientunitstayid','culturetakenoffset','culturesite','organism'],'e:')
    frame['patient_id'] = 'e:' + frame.uniquepid
    frame['site'] = frame.culturesite
    frame['hospital_id'] = frame.hospitalid.astype(str)
    frame['pre_icu'] = frame.culturetakenoffset < 0
    return finalize(frame[['episode_id','patient_id','species','drug','sir','site','hospital_id','pre_icu']],'eicu')


def summarize(frame):
    return {'cells':len(frame),'episodes':frame.episode_id.nunique(),
            'patients':frame.patient_id.nunique(),'hospitals':frame.hospital_id.nunique(),
            'pre_icu_episodes':frame.loc[frame.pre_icu,'episode_id'].nunique(),
            'species_episodes':frame.drop_duplicates('episode_id').species.value_counts().to_dict(),
            'sir':frame.sir.value_counts().to_dict()}


def main():
    OUTPUT.mkdir(parents=True,exist_ok=True)
    start=time.time()
    datasets={'mimic':mimic(),'eicu':eicu()}
    common=sorted(set(datasets['mimic'].drug) & set(datasets['eicu'].drug))
    summary={'common_drugs':common,'audit':AUDIT,'datasets':{}}
    for name,frame in datasets.items():
        frame.to_csv(OUTPUT/f'{name}_all_named_01.csv.gz',index=False)
        shared=frame[frame.drug.isin(common)].copy()
        shared=shared[shared.groupby('episode_id').drug.transform('nunique')>=5]
        shared.to_csv(OUTPUT/f'{name}_common_01.csv.gz',index=False)
        summary['datasets'][name]={'all_named':summarize(frame),'common':summarize(shared)}
        (shared.groupby(['species','drug','sir']).agg(cells=('episode_id','size'),patients=('patient_id','nunique'))
         .reset_index().to_csv(OUTPUT/f'{name}_feasibility_01.csv',index=False))
    manifest={}
    for name in ['microbiologyevents','icustays','patients','admissions','microLab','patient']:
        source=SOURCE/f'{name}.csv.gz'
        manifest[name]={'path':str(source),'bytes':source.stat().st_size,'sha256':sha256(source)}
    manifest['script_sha256']=sha256(Path(__file__))
    manifest['elapsed_seconds']=time.time()-start
    (OUTPUT/'source_manifest_01.json').write_text(json.dumps(manifest,indent=2))
    (OUTPUT/'cohort_summary_01.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
    print(json.dumps(summary,ensure_ascii=False,indent=2),flush=True)

if __name__=='__main__':
    main()
'''

## Patient split and nested support / 환자 분할과 중첩 support

In [ ]:
INPUT_SOURCES['patient_splits.py'] = r'''
"""Patient split and nested support / 환자 분할과 중첩 support"""
from pathlib import Path
import hashlib
import json
import pandas as pd

HOME=Path(__file__).resolve().parents[1]
INPUT=HOME/'preprocessing/cohort'
OUTPUT=HOME/'preprocessing/splits'
BUDGETS=(10,25,50,100)
REPEATS=range(10)

def rank(value,salt):
    return hashlib.sha256(f'{salt}|{value}'.encode()).hexdigest()

def main():
    data=pd.read_csv(INPUT/'mimic_common_01.csv.gz')
    patients=sorted(data.patient_id.unique(),key=lambda v:rank(v,'cdld-antibiotic-20260918-patient-v01'))
    count=len(patients)
    cutoffs=[int(count*.60),int(count*.75),int(count*.85),count]
    labels=['discovery','development','adaptation','query']
    mapping={};start=0
    for label,end in zip(labels,cutoffs):
        mapping.update({patient:label for patient in patients[start:end]});start=end
    data['split']=data.patient_id.map(mapping)
    assert data.groupby('patient_id').split.nunique().max()==1
    data[['patient_id','split']].drop_duplicates().to_csv(OUTPUT/'patient_splits_01.csv.gz',index=False)
    data.to_csv(OUTPUT/'mimic_split_cells_01.csv.gz',index=False)
    discovery=data[data.split=='discovery']
    candidates=[]; supports=[]
    for (species,drug),frame in discovery.groupby(['species','drug']):
        if frame.patient_id.nunique()<200:continue
        if min(frame.loc[frame.y==label,'patient_id'].nunique() for label in (0,1))<30:continue
        pool=data[(data.split=='adaptation')&(data.species==species)&(data.drug==drug)].copy()
        query=data[(data.split=='query')&(data.species==species)&(data.drug==drug)]
        candidates.append({'species':species,'target_drug':drug,'discovery_patients':frame.patient_id.nunique(),
                           'support_patients':pool.patient_id.nunique(),'query_patients':query.patient_id.nunique(),
                           'available_budgets':[b for b in BUDGETS if b<=pool.patient_id.nunique()]})
        for repeat in REPEATS:
            ordered=pool.assign(order=pool.episode_id.map(lambda v:rank(v,f'episode-{repeat}'))).sort_values('order').drop_duplicates('patient_id')
            ordered=ordered.assign(order=ordered.patient_id.map(lambda v:rank(v,f'patient-{repeat}'))).sort_values('order')
            for budget in BUDGETS:
                if len(ordered)<budget:continue
                subset=ordered.head(budget)
                assert subset.patient_id.nunique()==budget
                for position,row in enumerate(subset.itertuples()):
                    supports.append(dict(species=species,target_drug=drug,repeat=repeat,budget=budget,position=position,
                                         patient_id=row.patient_id,episode_id=row.episode_id))
    supports=pd.DataFrame(supports)
    supports.to_csv(OUTPUT/'a2_support_manifest_01.csv.gz',index=False)
    for _,frame in supports.groupby(['species','target_drug','repeat']):
        previous=set()
        for budget,subset in frame.groupby('budget',sort=True):
            current=set(subset.patient_id)
            assert previous<=current and len(current)==budget
            assert all(mapping[p]=='adaptation' for p in current)
            previous=current
    split_counts=data.groupby('split').agg(patients=('patient_id','nunique'),episodes=('episode_id','nunique'),cells=('y','size')).to_dict('index')
    report={'patient_overlap':0,'nested_support_check':'PASS','split_counts':split_counts,'candidates':candidates,
            'input_sha256':hashlib.sha256((INPUT/'mimic_common_01.csv.gz').read_bytes()).hexdigest(),
            'script_sha256':hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}
    (OUTPUT/'split_summary_01.json').write_text(json.dumps(report,indent=2))
    print(json.dumps(report,indent=2))

if __name__=='__main__':main()
'''

## Development patient roles / 개발 환자 역할

In [ ]:
INPUT_SOURCES['development_support.py'] = r'''
"""Development patient roles / 개발 환자 역할"""
from pathlib import Path
import pandas as pd
import hashlib,json
H=Path(__file__).resolve().parents[1];O=H/'preprocessing/development'
def rank(value,salt):return hashlib.sha256(f'{salt}|{value}'.encode()).hexdigest()
def main():
 src=H/'preprocessing/splits/mimic_split_cells_01.csv.gz'
 data=pd.read_csv(src,usecols=['patient_id','episode_id','species','drug','split'])
 proposal=json.loads((O/'feasibility_proposal.json').read_text());dev=set(proposal['dev_drugs'])
                                                                        
 patients=sorted(data.loc[data.split=='development','patient_id'].unique(),key=lambda x:rank(x,'dev117-patient-role'))
 support=set(patients[:int(len(patients)*.70)])
 manifest=[];summaries=[]
 for task in proposal['rows']:
  species,target,drug=(task[k] for k in ['species','target','development_drug'])
  g=data[(data.split=='development')&(data.species==species)]
  obs=g[~g.drug.isin(dev|{target})];counts=obs.groupby('episode_id').drug.nunique();eligible=set(counts[counts>=4].index)
  goals=g[(g.drug==drug)&g.episode_id.isin(eligible)][['patient_id','episode_id']].drop_duplicates()
  a=goals[goals.patient_id.isin(support)];q=goals[~goals.patient_id.isin(support)]
  assert set(a.patient_id).isdisjoint(q.patient_id)
  summaries.append(dict(**task,support_pool=int(a.patient_id.nunique()),query_patients=int(q.patient_id.nunique()),budget100_available=a.patient_id.nunique()>=100))
  for repeat in range(10):
   ordered=a.assign(rank=a.episode_id.map(lambda x:rank(x,f'episode117-{repeat}'))).sort_values('rank').drop_duplicates('patient_id')
   ordered=ordered.assign(rank=ordered.patient_id.map(lambda x:rank(x,f'support117-{repeat}'))).sort_values('rank')
   for budget in [10,25,50,100]:
    if len(ordered)<budget:continue
    for position,row in enumerate(ordered.head(budget).itertuples()):
     manifest.append(dict(species=species,evaluation_drug=target,development_drug=drug,repeat=repeat,budget=budget,position=position,patient_id=row.patient_id,episode_id=row.episode_id))
 pd.DataFrame(manifest).to_csv(O/'candidate_support_manifest.csv.gz',index=False)
 (O/'candidate_patient_roles.json').write_text(json.dumps({'salt':'dev117-patient-role','support_fraction':.70,'patient_roles':{str(x):'support' if x in support else 'query' for x in patients}},indent=2))
 (O/'candidate_split_summary.json').write_text(json.dumps({'status':'proposal','all_budget100_available':all(x['budget100_available'] for x in summaries),'tasks':summaries,'target_labels_read':False,'source_sha256':hashlib.sha256(src.read_bytes()).hexdigest()},indent=2))
 print('tasks',len(summaries),'budget100 available',sum(x['budget100_available'] for x in summaries),'min support',min(x['support_pool'] for x in summaries),'min query',min(x['query_patients'] for x in summaries),flush=True)
if __name__=='__main__':main()
'''

## Excluded-drug training inputs / 목표약·개발약 제외 입력

In [ ]:
INPUT_SOURCES['training_inputs.py'] = r'''
"""Excluded-drug training inputs / 목표약·개발약 제외 입력"""
from pathlib import Path
import json,hashlib
import numpy as np
import pandas as pd
H=Path(__file__).resolve().parents[1];O=H/'preprocessing/training';P=H/'preprocessing/development'
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
def write(p,x):p.write_text(json.dumps(x,ensure_ascii=False,indent=2))
def rank(x,salt):return hashlib.sha256(f'{salt}|{x}'.encode()).hexdigest()
def main():
 cfg=json.loads((O/'frozen_protocol.json').read_text());dev=set(cfg['development_drugs'])
 src=H/'preprocessing/splits/mimic_split_cells_01.csv.gz'
 d=pd.read_csv(src);d=d[d.split.isin(['discovery','development'])].copy()
 assert d.groupby('patient_id').split.nunique().max()==1
 proposal=json.loads((P/'candidate_split_summary.json').read_text())['tasks']
 roles=json.loads((P/'candidate_patient_roles.json').read_text())['patient_roles']
 sm=pd.read_csv(P/'candidate_support_manifest.csv.gz')
 jobs=[];counts=[]
 for species,target in sorted(set((r['species'],r['target']) for r in proposal)):
  folder=O/'models'/f"{species}__{target.replace('/','_')}";folder.mkdir(parents=True,exist_ok=True)
  g=d[(d.species==species)&(d.split=='discovery')&~d.drug.isin(dev|{target})].sort_values(['episode_id','drug']).copy()
                                                                                 
  ec=g.groupby('episode_id').drug.nunique();g=g[g.episode_id.isin(ec[ec>=2].index)].reset_index(drop=True)
  episodes=sorted(g.episode_id.unique());drugs=sorted(g.drug.unique());em={x:i for i,x in enumerate(episodes)};dm={x:i for i,x in enumerate(drugs)}
  held=[min(z.index,key=lambda i:rank(f"{ep}|{g.loc[i,'drug']}",'full-internal-v1')) for ep,z in g.groupby('episode_id')]
  mask=g.index.isin(held);x=np.column_stack([g.episode_id.map(em),g.drug.map(dm)]).astype('int32');y=g.y.to_numpy('float32')
  np.savez_compressed(folder/'split.npz',train_x=x[~mask],train_y=y[~mask],val_x=x[mask],val_y=y[mask])
  write(folder/'entities.json',dict(episodes=episodes,drugs=drugs,excluded_drugs=sorted(dev|{target})))
  for task in [r for r in proposal if r['species']==species and r['target']==target]:
   drug=task['development_drug'];a=d[(d.species==species)&(d.split=='development')]
   known=a[a.drug.isin(drugs)];n=known.groupby('episode_id').drug.nunique();eligible=set(n[n>=4].index)
   goals=a[(a.drug==drug)&a.episode_id.isin(eligible)].sort_values(['patient_id','episode_id'])
   support=goals[goals.patient_id.map(lambda v:roles[str(v)]=='support')];query=goals[goals.patient_id.map(lambda v:roles[str(v)]=='query')]
   assert set(support.patient_id).isdisjoint(query.patient_id)
   tasksupport=sm[(sm.species==species)&(sm.evaluation_drug==target)&(sm.development_drug==drug)]
   all_support=sorted(tasksupport.episode_id.unique());qe=list(query.episode_id);new=all_support+qe;assert len(set(new))==len(new)
   newmap={ep:len(episodes)+i for i,ep in enumerate(new)}
   ox=[];oy=[]
   for ep in new:
    rows=known[known.episode_id==ep].copy();rows['order']=rows.drug.map(lambda v:rank(f'{ep}|{v}','dev118-k4'));rows=rows.sort_values('order').head(4)
    assert len(rows)==4
    for row in rows.itertuples():ox.append([newmap[ep],dm[row.drug]]);oy.append(row.y)
   labelmap=dict(zip(support.episode_id,support.y));fits={};labels={}
   for repeat in cfg['repeats'] if isinstance(cfg['repeats'],list) else range(cfg['repeats']):
    for b in cfg['support_budgets']:
     chosen=tasksupport[(tasksupport['repeat']==repeat)&(tasksupport.budget==b)].sort_values('position');assert len(chosen)==b
     fits[f'x_{repeat}_{b}']=np.array([[newmap[e],len(drugs)] for e in chosen.episode_id],np.float32)
     labels[f'y_{repeat}_{b}']=np.array([labelmap[e] for e in chosen.episode_id],np.float32)
   path=folder/('dev_'+drug.replace('/','_'));path.mkdir(exist_ok=True)
   np.savez_compressed(path/'adapt_inputs.npz',observed_x=np.array(ox,np.float32),observed_y=np.array(oy,np.float32),query_x=np.array([[newmap[e],len(drugs)] for e in qe],np.float32),new_rows=len(new),**fits,**labels)
   np.savez_compressed(path/'query_labels.npz',y=query.y.to_numpy('float32'))
   write(path/'index.json',dict(support_episodes=all_support,query_episodes=qe,query_patients=query.patient_id.tolist(),development_drug=drug))
   counts.append(dict(species=species,evaluation_drug=target,development_drug=drug,query_patients=int(query.patient_id.nunique()),S=int(query[query.y==1].patient_id.nunique()),nonS=int(query[query.y==0].patient_id.nunique())))
  for seed in cfg['seed']:jobs.append(dict(folder=str(folder),species=species,target=target,seed=seed))
 write(O/'jobs.json',jobs);write(O/'class_counts.json',counts)
 write(O/'input_manifest.json',{str(f.relative_to(H)):sha(f) for f in O.glob('models/**/*') if f.is_file()})
 write(O/'preparation_audit.json',dict(passed=True,jobs=len(jobs),development_combinations=len(set((r['species'],r['development_drug']) for r in counts)),source_sha256=sha(src),low_class_tasks=[r for r in counts if min(r['S'],r['nonS'])<10]))
 print('prepared',len(jobs),'jobs',len(counts),'dev tasks',flush=True)
if __name__=='__main__':main()
'''

## Evaluation episode manifest / 평가 사례 목록

In [ ]:
INPUT_SOURCES['evaluation_manifest.py'] = r'''
"""Evaluation episode manifest / 평가 사례 목록"""
from pathlib import Path
import json,hashlib
import pandas as pd
H=Path(__file__).resolve().parents[1];D=H/'preprocessing/training';O=H/'preprocessing/evaluation'
def rank(x,s):return hashlib.sha256(f'{s}|{x}'.encode()).hexdigest()
def main():
 O.mkdir(exist_ok=True);src=H/'preprocessing/splits/mimic_split_cells_01.csv.gz';data=pd.read_csv(src,usecols=['patient_id','episode_id','species','drug','split']);supports=[];queries=[];observations=[]
 for folder in sorted((D/'models').iterdir()):
  entity=json.loads((folder/'entities.json').read_text());species=folder.name.split('__')[0];target=next(x for x in entity['excluded_drugs'] if x not in ['ceftazidime','gentamicin','clindamycin'])
  g=data[data.species==species];known=g[g.drug.isin(entity['drugs'])];n=known.groupby('episode_id').drug.nunique();eligible=set(n[n>=4].index)
  targetrows=g[(g.drug==target)&g.episode_id.isin(eligible)];pool=targetrows[targetrows.split=='adaptation'];query=targetrows[targetrows.split=='query'].sort_values(['patient_id','episode_id'])
  assert pool.patient_id.nunique()>=100 and set(pool.patient_id).isdisjoint(query.patient_id)
  for r in range(10):
   ordered=pool.assign(order=pool.episode_id.map(lambda x:rank(x,f'episode-{r}'))).sort_values('order').drop_duplicates('patient_id');ordered=ordered.assign(order=ordered.patient_id.map(lambda x:rank(x,f'patient-{r}'))).sort_values('order')
   for budget in [10,25,50,100]:
    for pos,row in enumerate(ordered.head(budget).itertuples()):supports.append(dict(species=species,target=target,repeat=r,budget=budget,position=pos,patient_id=row.patient_id,episode_id=row.episode_id))
  for row in query.itertuples():queries.append(dict(species=species,target=target,patient_id=row.patient_id,episode_id=row.episode_id))
  required={x['episode_id'] for x in supports if x['species']==species and x['target']==target}|set(query.episode_id)
  for ep,rows in known[known.episode_id.isin(required)].groupby('episode_id'):
                                                                        
   rows=rows.assign(order=rows.drug.map(lambda x:rank(f'{ep}|{x}','dev118-k4'))).sort_values('order').head(4);assert len(rows)==4
   for row in rows.itertuples():observations.append(dict(species=species,target=target,episode_id=ep,observed_drug=row.drug))
 for name,rows in [('support',supports),('query',queries),('observations',observations)]:pd.DataFrame(rows).to_csv(O/(name+'_manifest.csv.gz'),index=False)
 (O/'audit.json').write_text(json.dumps(dict(passed=True,target_labels_read=False,combinations=28,source_sha256=hashlib.sha256(src.read_bytes()).hexdigest(),patient_split_unchanged=True,support_policy='same original hash ranking restricted to currently k4-eligible episodes'),indent=2))
 print('evaluation manifests complete',len(supports),len(queries),flush=True)
if __name__=='__main__':main()
'''

## Indexed AST arrays / AST 인덱스 배열

In [ ]:
INPUT_SOURCES['evaluation_arrays.py'] = r'''
"""Indexed AST arrays / AST 인덱스 배열"""
from pathlib import Path
import json,hashlib
import numpy as np
import pandas as pd
H=Path(__file__).resolve().parents[1];D=H/'preprocessing/training';W=H/'preprocessing/evaluation'
def rank(x):return hashlib.sha256(f'dev118-k4|{x}'.encode()).hexdigest()
def main():
 internal=H/'preprocessing/splits/mimic_split_cells_01.csv.gz';external=H/'preprocessing/cohort/eicu_common_01.csv.gz'
 data=pd.read_csv(internal);ext=pd.read_csv(external);support=pd.read_csv(W/'support_manifest.csv.gz');query=pd.read_csv(W/'query_manifest.csv.gz');observed=pd.read_csv(W/'observations_manifest.csv.gz');audit=[]
 for folder in sorted((D/'models').iterdir()):
  entity=json.loads((folder/'entities.json').read_text());species=folder.name.split('__')[0];target=next(x for x in entity['excluded_drugs'] if x not in ['ceftazidime','gentamicin','clindamycin']);nu=len(entity['episodes']);ni=len(entity['drugs']);drugs={d:i for i,d in enumerate(entity['drugs'])}
  s=support[(support.species==species)&(support.target==target)];q=query[(query.species==species)&(query.target==target)];obs=observed[(observed.species==species)&(observed.target==target)]
  episodes=sorted(set(s.episode_id)|set(q.episode_id));ids={ep:nu+i for i,ep in enumerate(episodes)};assert set(episodes).isdisjoint(entity['episodes']);assert set(s.patient_id).isdisjoint(q.patient_id)
  known=data[(data.species==species)&data.drug.isin(drugs)][['episode_id','drug','y']]
  joined=obs.merge(known,left_on=['episode_id','observed_drug'],right_on=['episode_id','drug'],validate='many_to_one');assert len(joined)==len(obs);assert joined.groupby('episode_id').size().eq(4).all()
  arrays=dict(observed_x=np.array([[ids[r.episode_id],drugs[r.drug]] for r in joined.itertuples()],dtype='float32'),observed_y=joined.y.to_numpy(dtype='float32'),query_x=np.array([[ids[ep],ni] for ep in q.episode_id],dtype='float32'),new_rows=np.int64(len(episodes)))
                                                                     
  sy=data[(data.species==species)&(data.drug==target)&(data.split=='adaptation')][['episode_id','y']].set_index('episode_id').y
  for rep in range(10):
   previous=set()
   for b in [10,25,50,100]:
    rows=s[(s.repeat==rep)&(s.budget==b)].sort_values('position');assert len(rows)==b and rows.patient_id.nunique()==b;assert previous<=set(rows.episode_id);previous=set(rows.episode_id)
    arrays[f'x_{rep}_{b}']=np.array([[ids[ep],ni] for ep in rows.episode_id],dtype='float32');arrays[f'y_{rep}_{b}']=sy.loc[rows.episode_id].to_numpy(dtype='float32')
  out=W/'models'/folder.name;out.mkdir(parents=True,exist_ok=True);np.savez_compressed(out/'mimic_adapt_inputs.npz',**arrays)
  (out/'mimic_index.json').write_text(json.dumps(dict(species=species,target=target,episodes=episodes,query_episodes=q.episode_id.tolist(),query_patients=q.patient_id.tolist(),target_query_labels_exported=False),indent=2))
  g=ext[ext.species==species];known=g[g.drug.isin(drugs)];counts=known.groupby('episode_id').drug.nunique();eligible=set(counts[counts>=4].index);eq=g[(g.drug==target)&g.episode_id.isin(eligible)][['patient_id','episode_id']].sort_values(['patient_id','episode_id']);assert not eq.episode_id.duplicated().any();eps=eq.episode_id.tolist();eids={ep:nu+i for i,ep in enumerate(eps)};ex=[];ey=[]
  for ep,rows in known[known.episode_id.isin(eps)].groupby('episode_id'):
   rows=rows.assign(order=rows.drug.map(lambda drug:rank(f'{ep}|{drug}'))).sort_values('order').head(4);assert len(rows)==4
   for r in rows.itertuples():ex.append([eids[ep],drugs[r.drug]]);ey.append(r.y)
  np.savez_compressed(out/'eicu_observed_inputs.npz',observed_x=np.array(ex,dtype='float32').reshape(-1,2),observed_y=np.array(ey,dtype='float32'),query_x=np.array([[eids[ep],ni] for ep in eps],dtype='float32').reshape(-1,2),new_rows=np.int64(len(eps)))
  (out/'eicu_index.json').write_text(json.dumps(dict(species=species,target=target,query_episodes=eps,query_patients=eq.patient_id.tolist(),target_query_labels_exported=False),indent=2))
  audit.append(dict(species=species,target=target,internal_queries=len(q),external_queries=len(eq),external_patients=eq.patient_id.nunique(),known_drugs=ni,passed=True))
 manifest={str(f.relative_to(W)):hashlib.sha256(f.read_bytes()).hexdigest() for f in (W/'models').rglob('*') if f.is_file()}
 (W/'input_pack_audit.json').write_text(json.dumps(dict(passed=True,models=audit,query_outcomes_exported=False,eicu_target_fit=False,internal_source_sha256=hashlib.sha256(internal.read_bytes()).hexdigest(),external_source_sha256=hashlib.sha256(external.read_bytes()).hexdigest()),indent=2))
 (W/'input_pack_manifest.json').write_text(json.dumps(manifest,indent=2));print('locked evaluation packs',len(audit),'external eligible',sum(x['external_queries']>0 for x in audit))
if __name__=='__main__':main()
'''

## Generate the cohort and arrays / 코호트와 배열 생성

In [ ]:
def run_preprocessing(name, overrides):
    path = OUTPUT/'preprocessing/source'/name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(INPUT_SOURCES[name])
    namespace = {'__file__':str(path), '__name__':'preprocessing_module'}
    exec(compile(INPUT_SOURCES[name],str(path),'exec'), namespace)
    namespace.update(overrides)
    start = time.monotonic()
    with contextlib.redirect_stdout(io.StringIO()): namespace['main']()
    print(f"{name}: complete ({time.monotonic()-start:.1f} s)", flush=True)
run_preprocessing('cohort.py',dict(HOME=GENERATED,SOURCE=RAW_DIR,OUTPUT=STAGES[1]))
run_preprocessing('patient_splits.py',dict(HOME=GENERATED,INPUT=STAGES[1],OUTPUT=STAGES[2]))
cohort = json.loads((STAGES[1]/'cohort_summary_01.json').read_text())
splits = json.loads((STAGES[2]/'split_summary_01.json').read_text())
assert splits['patient_overlap']==0
assert {k:v['patients'] for k,v in splits['split_counts'].items()} == {'discovery':4207,'development':1052,'adaptation':702,'query':1052}
display(pd.DataFrame({k:v['common'] for k,v in cohort['datasets'].items()}).loc[['patients','episodes','cells']])
display(pd.DataFrame(splits['split_counts']).T)

run_preprocessing('development_support.py',dict(H=GENERATED,O=STAGES[117]))
run_preprocessing('training_inputs.py',dict(H=GENERATED,O=STAGES[118],P=STAGES[117]))
run_preprocessing('evaluation_manifest.py',dict(H=GENERATED,D=STAGES[118],O=STAGES[119]))
run_preprocessing('evaluation_arrays.py',dict(H=GENERATED,D=STAGES[118],W=STAGES[119]))
audit = json.loads((STAGES[119]/'input_pack_audit.json').read_text())
assert audit['passed'] and len(audit['models'])==28
assert not audit['query_outcomes_exported'] and not audit['eicu_target_fit']
display(pd.DataFrame(audit['models'])[['species','target','known_drugs','internal_queries','external_queries']])

## Standard-classifier features / 표준 분류기 입력
Each feature vector concatenates AST values and observation masks. / AST 값과 관측 마스크를 이어 붙입니다.

In [ ]:
from pathlib import Path
import numpy as np, json, hashlib, platform
ROOT=GENERATED
OUT=ROOT/'preprocessing/classical'
(OUT/'packs').mkdir(exist_ok=True)
DEV=ROOT/'preprocessing/training/models'
EVAL=ROOT/'preprocessing/evaluation/models'
def sha(f): return hashlib.sha256(f.read_bytes()).hexdigest()
def encode(z,nu,ni):
    ox=z['observed_x'].astype(int);oy=z['observed_y'];n=int(z['new_rows'])
    assert np.array_equal(ox,z['observed_x'])
    assert set(np.unique(oy))<= {0,1}
    assert np.all((ox[:,0]>=nu)&(ox[:,0]<nu+n)&(ox[:,1]>=0)&(ox[:,1]<ni))
    assert len(np.unique(ox,axis=0))==len(ox)
    X=np.zeros((n,2*ni),np.float32)
    X[ox[:,0]-nu,ox[:,1]]=oy;X[ox[:,0]-nu,ni+ox[:,1]]=1
    assert np.all(X[:,ni:].sum(1)==4)
    np.testing.assert_array_equal(X[ox[:,0]-nu,ox[:,1]],oy)
    np.testing.assert_array_equal(np.argwhere(X[:,ni:]) , np.array(sorted((ox-[nu,0]).tolist())))
    return X
records=[];sources={};audits=[]
for model in sorted(DEV.iterdir()):
    if not model.is_dir():continue
    ef=model/'entities.json';ent=json.loads(ef.read_text());nu=len(ent['episodes']);ni=len(ent['drugs']);sources[str(ef.relative_to(ROOT))]=sha(ef)
    assert not set(ent['excluded_drugs'])&set(ent['drugs'])
    for task in sorted(model.glob('dev_*')):
        ip=task/'adapt_inputs.npz';idxf=task/'index.json';yf=task/'query_labels.npz';idx=json.loads(idxf.read_text());z=np.load(ip)
        X=encode(z,nu,ni);q=z['query_x'].astype(int);ids=idx['support_episodes']+idx['query_episodes'];assert len(ids)==len(X)
        np.testing.assert_array_equal(q[:,0]-nu,np.arange(len(idx['support_episodes']),len(ids)))
        assert np.all(q[:,1]==ni)
        assert not {i.split('|')[0] for i in idx['support_episodes']}&set(idx['query_patients'])
        arrays={'X':X,'query_indices':q[:,0]-nu,'query_y':np.load(yf)['y']}
        for r in range(10):
            previous=[]
            for b in [10,25,50,100]:
                sx=z[f'x_{r}_{b}'].astype(int);sy=z[f'y_{r}_{b}'];assert len(sx)==b and np.all(sx[:,1]==ni)
                si=sx[:,0]-nu;assert len(set(si))==b and not set(si)&set(q[:,0]-nu)
                assert np.all((si>=0)&(si<len(idx['support_episodes'])))
                assert set(previous)<=set(si);previous=si
                arrays[f'idx_{r}_{b}']=si;arrays[f'y_{r}_{b}']=sy
        name=model.name+'--'+task.name;dest=OUT/'packs'/f'dev--{name}.npz';np.savez_compressed(dest,**arrays)
        records.append(dict(phase='dev',name=name,task=model.name.split('__')[0]+'__'+idx['development_drug'],pack=str(dest.relative_to(OUT)),n_query=len(q)))
        for f in [ip,idxf,yf]:sources[str(f.relative_to(ROOT))]=sha(f)
        audits.append(dict(name=name,phase='dev',rows=len(ids),observations=len(z['observed_x']),identity_and_k4_passed=True))
    ev=EVAL/model.name
    if not ev.exists():continue
    mi=json.loads((ev/'mimic_index.json').read_text());z=np.load(ev/'mimic_adapt_inputs.npz');X=encode(z,nu,ni);ids=mi['episodes'];assert len(ids)==len(X)
    q=z['query_x'].astype(int);assert [ids[i] for i in q[:,0]-nu]==mi['query_episodes'];assert np.all(q[:,1]==ni)
    arrays={'X':X,'query_indices':q[:,0]-nu}
    for r in range(10):
        previous=[]
        for b in [10,25,50,100]:
            sx=z[f'x_{r}_{b}'].astype(int);si=sx[:,0]-nu;assert len(set(si))==b and np.all(sx[:,1]==ni)
            assert not {ids[i].split('|')[0] for i in si}&set(mi['query_patients'])
            assert not set(si)&set(q[:,0]-nu);assert set(previous)<=set(si);previous=si
            arrays[f'idx_{r}_{b}']=si;arrays[f'y_{r}_{b}']=z[f'y_{r}_{b}']
    ez=np.load(ev/'eicu_observed_inputs.npz');ei=json.loads((ev/'eicu_index.json').read_text());EX=encode(ez,nu,ni);eq=ez['query_x'].astype(int)
    assert np.all(eq[:,1]==ni);assert len(EX)==len(ei['query_episodes']);np.testing.assert_array_equal(eq[:,0]-nu,np.arange(len(EX)))
    arrays['external_X']=EX[eq[:,0]-nu]
    dest=OUT/'packs'/f'eval--{model.name}.npz';np.savez_compressed(dest,**arrays)
    records.append(dict(phase='eval',name=model.name,task=model.name,pack=str(dest.relative_to(OUT)),n_query=len(q),n_external=len(EX)))
    for f in ev.iterdir():sources[str(f.relative_to(ROOT))]=sha(f)
    audits.append(dict(name=model.name,phase='eval',rows=len(ids),observations=len(z['observed_x']),identity_and_k4_passed=True))
assert sum(r['phase']=='dev' for r in records)==51
assert len({r['task'] for r in records if r['phase']=='dev'})==7
assert sum(r['phase']=='eval' for r in records)==28
(OUT/'records.json').write_text(json.dumps(records,indent=2))
(OUT/'input_source_hashes.json').write_text(json.dumps(sources,indent=2))
(OUT/'input_audit.json').write_text(json.dumps({'passed':True,'development_packs':51,'development_tasks':7,'evaluation_packs':28,'audits':audits,'final_query_labels_exported':False},indent=2))
(OUT/'pack_hashes.json').write_text(json.dumps({str(f.relative_to(OUT)):sha(f) for f in (OUT/'packs').glob('*.npz')},indent=2))
print('Prepared 28 evaluation feature packs and development inputs.')
print('All support/query disjointness, nested budgets, binary labels, excluded columns and external row order passed.')
print('Final query labels are absent from exported packs.')

## Connect inputs and separate query labels / 모델 입력과 query 정답 연결
Support 10/20/30/40/50 use prefixes of one ordering. Query labels are stored separately for scoring. / 같은 순서의 앞부분으로 support를 구성하며 query 정답은 채점용으로 분리합니다.

In [ ]:
BUNDLE = RUN
BUNDLE.mkdir(exist_ok=True)
protocol = json.loads((PACKAGE/'config/protocol.json').read_text())
contexts = protocol['contexts']
selected_dev = set()
assert len(contexts)==28
for name in contexts:
    dest = BUNDLE/'inputs'/name
    dest.mkdir(parents=True,exist_ok=True)
    model = STAGES[118]/'models'/name
    for item in model.iterdir():
        if item.is_file(): shutil.copy2(item,dest/item.name)
        elif (name,item.name) in selected_dev:
            shutil.copytree(item,dest/item.name,dirs_exist_ok=True)
            info=json.loads((dest/item.name/'index.json').read_text())
            (dest/item.name/'index.json').write_text(json.dumps({'development_drug':info['development_drug']},indent=2))
    for item in (STAGES[119]/'models'/name).iterdir(): shutil.copy2(item,dest/item.name)
    f = dest/'mimic_adapt_inputs.npz'
    with np.load(f) as z: arrays = dict(z)
    for repeat in range(10):
        for b in [20,30,40]:
            for prefix in ['x','y']: arrays[f'{prefix}_{repeat}_{b}'] = arrays[f'{prefix}_{repeat}_50'][:b]
    np.savez_compressed(f,**arrays)

records = [r for r in json.loads((STAGES[128]/'records.json').read_text()) if r['phase']=='eval']
(BUNDLE/'classical/packs').mkdir(parents=True,exist_ok=True)
(BUNDLE/'classical/records.json').write_text(json.dumps(records,indent=2))
shutil.copy2(PACKAGE/'config/classical.json',BUNDLE/'classical/selected_settings.json')
for rec in records:
    with np.load(STAGES[128]/rec['pack']) as z: arrays = dict(z)
    for repeat in range(10):
        for b in [20,30,40]:
            for prefix in ['idx','y']: arrays[f'{prefix}_{repeat}_{b}'] = arrays[f'{prefix}_{repeat}_50'][:b]
    np.savez_compressed(BUNDLE/'classical'/rec['pack'],**arrays)

raw = {'mimic':pd.read_csv(STAGES[2]/'mimic_split_cells_01.csv.gz'),
       'eicu':pd.read_csv(STAGES[1]/'eicu_common_01.csv.gz')}
counts = {}
for dataset,source in raw.items():
    patient_ids,episode_ids=set(),set()
    for name in contexts:
        meta=json.loads((STAGES[119]/'models'/name/f'{dataset}_index.json').read_text())
        identity=json.loads((STAGES[119]/'models'/name/'mimic_index.json').read_text())
        frame=source[(source.species==identity['species']) & (source.drug==identity['target'])].set_index('episode_id').loc[meta['query_episodes']].reset_index()
        assert frame.patient_id.tolist()==meta['query_patients']
        out=BUNDLE/'labels'/dataset/name;out.mkdir(parents=True,exist_ok=True)
        frame[['patient_id','episode_id','y','sir']].to_csv(out/'scoring_index.csv.gz',index=False)
        patient_ids.update(frame.patient_id);episode_ids.update(frame.episode_id)
    counts[dataset]=dict(patients=len(patient_ids),episodes=len(episode_ids))
assert counts=={'mimic':{'patients':992,'episodes':1778},'eicu':{'patients':155,'episodes':218}}
display(pd.DataFrame(counts).T)
print('Input generation complete. Continue with 02_models.ipynb.')